# Step 1: Data Loading and normalization

In [1]:
# Install sklearn if not already installed
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Install pywt if not already installed
%pip install PyWavelets

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Install tensorflow — use tensorflow-macos + tensorflow-metal on Apple Silicon (macOS)
# This fixes the RuntimeError that occurs with the generic tensorflow build on macOS ARM
%pip install tensorflow-macos tensorflow-metal

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow-macos (from versions: none)
ERROR: No matching distribution found for tensorflow-macos


In [4]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np

# Paths to your data folders
# Make sure you are running this notebook in the same directory as the data folders
truth_path = '../LieWaves/Truth_Sessions/1_BandPass_Filtered/'
lie_path = '../LieWaves/Lie_Sessions/1_BandPass_Filtered/'

# Function to load all CSVs and concatenate them into a single DataFrame
def load_and_concatenate_data(path):
    data_frames = []
    for filename in os.listdir(path):
        if filename.endswith('.csv'):
            df = pd.read_csv(os.path.join(path, filename))
            data_frames.append(df)
    concatenated_df = pd.concat(data_frames, ignore_index=True)
    return concatenated_df

truth_data = load_and_concatenate_data(truth_path).values  # keep as raw numpy
lie_data = load_and_concatenate_data(lie_path).values      # scaling happens after the split



# Step 2: Segmenting the Data

In [5]:
# Function to segment data
def segment_data(df, window_size=128, overlap=64):
    segments = []
    for start in range(0, len(df) - window_size, overlap):
        segment = df[start:start + window_size, :]
        segments.append(segment)
    return np.array(segments)

# Segment the data
window_size = 128  # 1 second of data
overlap = 64  # 50% overlap

truth_segments = segment_data(truth_data, window_size, overlap)
lie_segments = segment_data(lie_data, window_size, overlap)

# Step 3: Feature Extraction with DWT

In [6]:
import pywt

# Function to extract DWT features
def extract_dwt_features(segments, wavelet='db4', level=4):
    features = []
    for segment in segments:
        segment_features = []
        for channel in range(segment.shape[1]):
            coeffs = pywt.wavedec(segment[:, channel], wavelet, level=level)
            coeffs_flat = np.hstack(coeffs)
            segment_features.append(coeffs_flat)
        features.append(np.hstack(segment_features))
    return np.array(features)

# Extract DWT features
truth_features = extract_dwt_features(truth_segments)
lie_features = extract_dwt_features(lie_segments)

# Step 4: Preparing Data for CNN

In [7]:
# Create labels: 1 for truth, 0 for lie
truth_labels = np.ones(truth_features.shape[0])
lie_labels = np.zeros(lie_features.shape[0])

# Combine features and labels
X = np.vstack((truth_features, lie_features))
y = np.hstack((truth_labels, lie_labels))

from sklearn.model_selection import train_test_split

# Step 1 — hold out 20% as the test set (never seen during training or model selection)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 2 — split the remaining 80% into 87.5% train / 12.5% val → ~70% / 10% of total
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp
)

# Step 3 — fit the scaler on training features ONLY, then transform val and test
# Fitting on val/test would let their statistics leak into preprocessing
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print("Train set class distribution:", np.bincount(y_train.astype(int)))
print("Val   set class distribution:", np.bincount(y_val.astype(int)))
print("Test  set class distribution:", np.bincount(y_test.astype(int)))

Train set class distribution: [2833 2833]
Val   set class distribution: [405 405]
Test  set class distribution: [810 810]


# Step 5: Define and Train the CNN

In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import callbacks as keras_callbacks

# Define the CNN model
model = models.Sequential()

# Convolution Layers
# Stage 1
model.add(layers.Conv1D(256, 3, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(tf.keras.layers.BatchNormalization())
model.add(layers.MaxPooling1D(2))
model.add(layers.Dropout(0.25))

# Stage 2
model.add(layers.Conv1D(128, 3, activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(layers.MaxPooling1D(2))
model.add(layers.Dropout(0.25))

# Stage 3
model.add(layers.Conv1D(64, 3, activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(layers.MaxPooling1D(2))
model.add(layers.Dropout(0.25))

# Flatten
model.add(layers.Flatten())

# Fully Connected Layers
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(64, activation='relu'))

# Output — single sigmoid unit for binary classification with binary_crossentropy
model.add(layers.Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Reshape data for the CNN
X_train_cnn = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_val_cnn   = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))
X_test_cnn  = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

# Define callbacks — use a different variable name to avoid shadowing the keras callbacks module
cb_list = [
    keras_callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras_callbacks.ModelCheckpoint('best_model.keras', save_best_only=True)
]

# Train using the validation set — X_test_cnn is never touched here
history = model.fit(X_train_cnn, y_train, epochs=10, batch_size=32, validation_data=(X_val_cnn, y_val), callbacks=cb_list)

# Load the best model weights
model = tf.keras.models.load_model('best_model.keras')

c:\Users\aroon\Desktop\Masterarbeit\.venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 34s 170ms/step - accuracy: 0.5427 - loss: 0.7653 - val_accuracy: 0.5716 - val_loss: 0.6839
Epoch 2/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 30s 170ms/step - accuracy: 0.6322 - loss: 0.6323 - val_accuracy: 0.4975 - val_loss: 0.9020
Epoch 3/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 30s 166ms/step - accuracy: 0.7091 - loss: 0.5871 - val_accuracy: 0.6469 - val_loss: 0.7392
Epoch 4/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 30s 167ms/step - accuracy: 0.7902 - loss: 0.4583 - val_accuracy: 0.7235 - val_loss: 0.5437
Epoch 5/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 30s 167ms/step - accuracy: 0.8475 - loss: 0.3558 - val_accuracy: 0.7407 - val_loss: 0.5518
Epoch 6/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 30s 166ms/step - accuracy: 0.8803 - loss: 0.2816 - val_accuracy: 0.7716 - val_loss: 0.5520
Epoch 7/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 30s 167ms/step - accuracy: 0.9252 - loss: 0.1948 - val_accuracy: 0.7444 - val_loss: 0.7237
Epoch 8/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 30s 168ms/step - accuracy: 0.9392 - loss: 0

# Step 6: Evaluate the Model

In [9]:
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Load the saved model (includes architecture + weights)
model = tf.keras.models.load_model('best_model.keras')

# Make predictions
y_pred = model.predict(X_test_cnn)
y_pred_classes = (y_pred > 0.5).astype("int32")

# Evaluate the model
print(classification_report(y_test, y_pred_classes, zero_division=1))
print("F1 Score:", f1_score(y_test, y_pred_classes, zero_division=1))


51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step
              precision    recall  f1-score   support

         0.0       0.71      0.75      0.73       810
         1.0       0.73      0.69      0.71       810

    accuracy                           0.72      1620
   macro avg       0.72      0.72      0.72      1620
weighted avg       0.72      0.72      0.72      1620

F1 Score: 0.7111959287531806
